# 05 - Poblacion indigena real (Censo 2020, INPI) vs. composicion de casos
Resuelve la limitacion metodologica de 04_indigenous_context.ipynb: compara
el % de casos de suicidio con autoadscripcion indigena (Conindig) contra el
% REAL de poblacion indigena de cada municipio -- no solo la composicion de
los casos por si sola.

Fuente poblacion: INPI, 'Poblacion indigena autoadscrita por municipio'
(muestra censal 2020). https://www.inpi.gob.mx/indicadores2020/

SUPUESTO A DOCUMENTAR: este dato es una fotografia fija de 2020, aplicada
como referencia para todo el periodo 2019-2024 de casos. Es razonable
porque la composicion etnica municipal cambia lentamente, pero es un
supuesto explicito, no un dato anual real.

In [ ]:
import pandas as pd


## 1. Cargar poblacion indigena INPI y quedarnos solo con los conteos reales
El archivo trae varias filas por municipio (Poblacion, Error estandar,
limites de confianza, coeficiente de variacion) -- solo nos sirve la fila
'Poblacion'.

In [ ]:
df_indig_raw = pd.read_excel('../data/raw/2-poblacion-indigena-autoadscrita-por-municipio-muestra-censal-2020.xlsx')
print(f'Filas crudas (5 por municipio): {len(df_indig_raw):,}')
print(df_indig_raw.columns.tolist())
df_indig_raw.head(10)


In [ ]:
df_indig = df_indig_raw[df_indig_raw['Estimador'] == 'Población'].copy()
print(f'Filas con conteos reales (1 por municipio): {len(df_indig):,}')

# Extraer codigo de municipio de 3 digitos, igual que en 02_population_merge.ipynb
df_indig['mun_codigo'] = (df_indig['Clave de municipio'] % 1000).astype(int).astype(str).str.zfill(3)

df_indig['pct_indigena'] = round(
    df_indig['Se considera indígena'] / df_indig['Población de 3 años y más'] * 100, 2
)
df_indig[['mun_codigo', 'Municipio', 'Población de 3 años y más', 'Se considera indígena', 'pct_indigena']].head(10)


## 2. Comparacion agregada: Sierra Tarahumara vs. resto
% de poblacion indigena REAL, vs. % de CASOS indigenas ya calculado en
04_indigenous_context.ipynb (58.3% sierra, 6.1% resto, entre casos con dato
conocido).

In [ ]:
SIERRA_TARAHUMARA_CITADOS = [
    'Balleza', 'Batopilas', 'Bocoyna', 'Carichí', 'Chínipas', 'Guachochi',
    'Guadalupe y Calvo', 'Guazapares', 'Guerrero', 'Maguarichi', 'Morelos',
    'Moris', 'Nonoava', 'Ocampo', 'Temósachic', 'Urique', 'Uruachi',
]

# Verificar nombres contra ESTE catalogo tambien (puede tener otra ortografia)
nombres_inpi = set(df_indig['Municipio'].unique())
sin_match = [n for n in SIERRA_TARAHUMARA_CITADOS if n not in nombres_inpi]
print(f'Sin match en catalogo INPI: {sin_match}')
for n in sin_match:
    candidatos = [real for real in nombres_inpi if n.split()[0].lower() in real.lower()]
    print(f'  "{n}" -> candidatos: {candidatos}')


In [ ]:
# Ajustar segun el resultado de la celda anterior antes de continuar
CORRECCIONES_NOMBRE_INPI = {
    'Batopilas': 'Batopilas de Manuel Gómez Morín',  # verificar
}
sierra_inpi = [CORRECCIONES_NOMBRE_INPI.get(n, n) for n in SIERRA_TARAHUMARA_CITADOS]
sierra_inpi_validado = [n for n in sierra_inpi if n in nombres_inpi]
print(f'Validados: {len(sierra_inpi_validado)}/{len(SIERRA_TARAHUMARA_CITADOS)}')

df_indig['es_sierra_tarahumara'] = df_indig['Municipio'].isin(sierra_inpi_validado)

resumen_poblacion = df_indig.groupby('es_sierra_tarahumara').agg(
    poblacion_3mas=('Población de 3 años y más', 'sum'),
    poblacion_indigena=('Se considera indígena', 'sum'),
)
resumen_poblacion['pct_indigena_real'] = round(
    resumen_poblacion['poblacion_indigena'] / resumen_poblacion['poblacion_3mas'] * 100, 1
)
resumen_poblacion


## 3. La comparacion que realmente importa
% de CASOS indigenas vs. % de POBLACION indigena, lado a lado. Si el % de
casos excede claramente el % de poblacion, hay evidencia de sobrerrepresentacion
real -- no solo composicion demografica esperada.

In [ ]:
# Valores ya calculados en 04_indigenous_context.ipynb (entre casos con dato conocido)
pct_casos_indigenas_sierra = 58.3
pct_casos_indigenas_resto = 6.1

comparacion = pd.DataFrame({
    'region': ['Sierra Tarahumara', 'Resto del estado'],
    'pct_poblacion_indigena': [
        resumen_poblacion.loc[True, 'pct_indigena_real'],
        resumen_poblacion.loc[False, 'pct_indigena_real'],
    ],
    'pct_casos_indigenas': [pct_casos_indigenas_sierra, pct_casos_indigenas_resto],
})
comparacion['razon_casos_vs_poblacion'] = round(
    comparacion['pct_casos_indigenas'] / comparacion['pct_poblacion_indigena'], 2
)
comparacion


## 4. Guardar


In [ ]:
df_indig.to_csv('../data/processed/poblacion_indigena_municipal_2020.csv', index=False, encoding='utf-8')
comparacion.to_csv('../data/processed/comparacion_casos_vs_poblacion_indigena.csv', index=False, encoding='utf-8')
print('Guardado.')


## 5. Hallazgos
_Documentar aqui: si razon_casos_vs_poblacion es cercana a 1.0 (composicion
esperada, sin sobrerrepresentacion) o notablemente mayor a 1.0
(sobrerrepresentacion real de casos indigenas mas alla de lo que explicaria
la demografia). Recordar que el 64% de Conindig no especificado (ver
04_indigenous_context.ipynb) sigue siendo una limitacion que hay que
mencionar explicitamente en cualquier conclusion._